In [6]:
import requests
import pandas as pd 
import os 

BASE_URL = "https://api.pax-db.org/v6"
API_KEY = "pk_live_5edfaae3.kZApedQ-VdXMPW_i7DceNMlGc-DWNElf2BFfrOAB6YQ"

headers = {
    "x-api-key": API_KEY
}

def paxdb_get(endpoint, params=None):
    url = f"{BASE_URL}{endpoint}"
    response = requests.get(url, headers=headers, params=params, timeout=30)
    response.raise_for_status()
    return response.json()

In [7]:
print(API_KEY)

pk_live_5edfaae3.kZApedQ-VdXMPW_i7DceNMlGc-DWNElf2BFfrOAB6YQ


In [2]:
import json 
availiable_tissues = paxdb_get(
    "/metadata/organs",
    params={
        "species": 9606,
    }
)
print(availiable_tissues.keys())
df_tissues = pd.DataFrame(availiable_tissues["data"])
df_tissues.head()

dict_keys(['message', 'data'])


,organ_id,organ_name
0,UBERON:0002369,ADRENAL_GLAND
1,UBERON:0007795,ASCITIC_FLUID
2,CL:0000236,B_CELL
3,UBERON:0000955,BRAIN
4,UBERON_0000956,CEREBRAL_CORTEX


In [7]:
lung_specific_datasets = paxdb_get(
    "/metadata/dataset",
    params={
        "species_id" : 9606,
        "organ" : "LUNG"
    }
)
print(lung_specific_datasets.keys())
lung_specific_datasets = pd.DataFrame(lung_specific_datasets["result"])
lung_specific_datasets.head()
lung_integrated = lung_specific_datasets[lung_specific_datasets["name"].str.contains("integrated", case=False)]
lung_integrated

dict_keys(['message', 'total_hits', 'result'])


,dataset_id,name,score,organ,coverage
7,3105119500,H.sapiens - Lung (Integrated),32.4,LUNG,67


In [7]:
dataset_meta = paxdb_get(
    "/metadata/dataset/detail/3105119500"
)
print(dataset_meta.keys())
n_abundances = dataset_meta["num_abundances"]
print(n_abundances)

dict_keys(['message', 'dataset_id', 'name', 'score', 'description', 'organ', 'integrated', 'coverage', 'publication_year', 'num_abundances', 'has_peptide_counts', 'filename', 'species_id'])
13214


In [10]:
import time
import pandas as pd

def get_dataset_abundances_batched(
    dataset_id,
    total_abundances,
    batch_size=1000,
    sort="-abundance",
    sleep=0.2
):
    all_batches = []

    for start in range(0, total_abundances, batch_size):
        end = min(start + batch_size, total_abundances)

        data = paxdb_get(
            f"/abundances/dataset/{dataset_id}",
            params={
                "start": start,
                "end": end,
                "sort": sort
            }
        )

        # PaxDB returns a list directly here
        if isinstance(data, list):
            batch = data
        elif isinstance(data, dict):
            batch = data.get("result", [])
        else:
            raise TypeError(f"Unexpected response type: {type(data)}")

        if not batch:
            raise RuntimeError(f"No results returned for start={start}, end={end}. Full response: {data}")

        batch_df = pd.DataFrame(batch)
        all_batches.append(batch_df)

        print(f"Downloaded rows {start} to {end}: {len(batch)} rows")

        time.sleep(sleep)

    df = pd.concat(all_batches, ignore_index=True)

    return df

In [11]:
dataset_id = 3105119500
total_abundances = 13214

df_abundances = get_dataset_abundances_batched(
    dataset_id=dataset_id,
    total_abundances=total_abundances,
    batch_size=1000,
    sort="-abundance"
)

print(df_abundances.shape)
print(df_abundances.head())

Downloaded rows 0 to 1000: 1000 rows
Downloaded rows 1000 to 2000: 1000 rows
Downloaded rows 2000 to 3000: 1000 rows
Downloaded rows 3000 to 4000: 1000 rows
Downloaded rows 4000 to 5000: 1000 rows
Downloaded rows 5000 to 6000: 1000 rows
Downloaded rows 6000 to 7000: 1000 rows
Downloaded rows 7000 to 8000: 1000 rows
Downloaded rows 8000 to 9000: 1000 rows
Downloaded rows 9000 to 10000: 1000 rows
Downloaded rows 10000 to 11000: 1000 rows
Downloaded rows 11000 to 12000: 1000 rows
Downloaded rows 12000 to 13000: 1000 rows
Downloaded rows 13000 to 13214: 214 rows
(13214, 6)
        id             string_id  abundance  rank preferred_name  \
0  6223315  9606.ENSP00000370010    55863.0     1         TMSB4X   
1  6215557  9606.ENSP00000295897    46912.0     2            ALB   
2  6230280  9606.ENSP00000494175    37127.0     3            HBB   
3  6217738  9606.ENSP00000322421    36141.0     4           HBA1   
4  6212679  9606.ENSP00000251595    32467.0     5           HBA2   

               